Importing packages

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    mean_squared_error,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import TransformedTargetRegressor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

import matplotlib.pyplot as plt
import joblib
from datetime import datetime

In [ ]:
user_path = "Users/piotr/"

In [ ]:
df = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_additional.csv")

Regressor - param search

In [ ]:
X = df.drop(columns='volume_total')
y = df['volume_total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"y_train range: [{y_train.min():.0f}, {y_train.max():.0f}]")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

In [ ]:
models = {
    "xgb": (
        XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1,
            random_state=2026
        ),
        {
            "n_estimators": randint(200, 1200),
            "max_depth": randint(3, 12),
            "learning_rate": uniform(0.01, 0.1),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "min_child_weight": randint(1, 10),
            "gamma": uniform(0, 2.0),
            "reg_alpha": uniform(0, 1.0),
            "reg_lambda": uniform(0, 1.0),
        },
    ),
    
    "lgbm": (
        LGBMRegressor(
            objective="regression",
            n_jobs=-1,
            random_state=2026
        ),
        {
            "n_estimators": randint(200, 1200),
            "max_depth": randint(-1, 12),
            "learning_rate": uniform(0.01, 0.1),
            "num_leaves": randint(16, 256),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "min_child_samples": randint(5, 50),
            "reg_alpha": uniform(0, 1.0),
            "reg_lambda": uniform(0, 1.0),
        },
    ),
    
    "catboost": (
        CatBoostRegressor(
            loss_function="RMSE",
            verbose=0,
            random_state=2026
        ),
        {
            "depth": randint(4, 10),
            "learning_rate": uniform(0.01, 0.1),
            "n_estimators": randint(200, 1200),
            "l2_leaf_reg": uniform(1, 10),
            "bagging_temperature": uniform(0, 5),
        },
    ),
    
    "rf": (
        RandomForestRegressor(
            n_jobs=-1,
            random_state=2026
        ),
        {
            "n_estimators": randint(200, 800),
            "max_depth": randint(3, 20),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", 0.5, 0.8],
        },
    ),
    
    "xgb_log": (
        TransformedTargetRegressor(
            regressor=XGBRegressor(
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=-1,
                random_state=2026
            ),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        {
            "regressor__n_estimators": randint(200, 1200),
            "regressor__max_depth": randint(3, 12),
            "regressor__learning_rate": uniform(0.01, 0.1),
            "regressor__subsample": uniform(0.6, 0.4),
            "regressor__colsample_bytree": uniform(0.6, 0.4),
            "regressor__min_child_weight": randint(1, 10),
            "regressor__gamma": uniform(0, 2.0),
            "regressor__reg_alpha": uniform(0, 1.0),
            "regressor__reg_lambda": uniform(0, 1.0),
        },
    ),
    
    "lgbm_log": (
        TransformedTargetRegressor(
            regressor=LGBMRegressor(
                objective="regression",
                n_jobs=-1,
                random_state=2026
            ),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        {
            "regressor__n_estimators": randint(200, 1200),
            "regressor__max_depth": randint(-1, 12),
            "regressor__learning_rate": uniform(0.01, 0.1),
            "regressor__num_leaves": randint(16, 256),
            "regressor__subsample": uniform(0.6, 0.4),
            "regressor__colsample_bytree": uniform(0.6, 0.4),
            "regressor__min_child_samples": randint(5, 50),
            "regressor__reg_alpha": uniform(0, 1.0),
            "regressor__reg_lambda": uniform(0, 1.0),
        },
    ),
    
    "catboost_log": (
        TransformedTargetRegressor(
            regressor=CatBoostRegressor(
                loss_function="RMSE",
                verbose=0,
                random_state=2026
            ),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        {
            "regressor__depth": randint(4, 10),
            "regressor__learning_rate": uniform(0.01, 0.1),
            "regressor__n_estimators": randint(200, 1200),
            "regressor__l2_leaf_reg": uniform(1, 10),
            "regressor__bagging_temperature": uniform(0, 5),
        },
    ),
    
    "rf_log": (
        TransformedTargetRegressor(
            regressor=RandomForestRegressor(
                n_jobs=-1,
                random_state=2026
            ),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        {
            "regressor__n_estimators": randint(200, 800),
            "regressor__max_depth": randint(3, 20),
            "regressor__min_samples_split": randint(2, 20),
            "regressor__min_samples_leaf": randint(1, 10),
            "regressor__max_features": ["sqrt", "log2", 0.5, 0.8],
        },
    ),
}

print(f"Zdefiniowano {len(models)} modeli regresji")

In [ ]:
results = {}

for name, (estimator, param_dist) in models.items():
    print(f"\n{'='*60}\n=== {name.upper()} ===\n{'='*60}")
    
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=30,  # zwiększono z 20
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        random_state=2026,
        verbose=2
    )
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    
    results[name] = {
        "search": search,
        "best_model": best_model,
        "best_params": search.best_params_,
        "cv_best_score": -search.best_score_,
        "test_rmse": rmse,
    }
    
    print(f"\n✓ Best params: {search.best_params_}")
    print(f"CV RMSE: {-search.best_score_:.2f}")
    print(f"Test RMSE: {rmse:.2f}")

In [ ]:
best_name = min(results, key=lambda k: results[k]["test_rmse"])
best_model = results[best_name]["best_model"]

print(f"\n{'='*60}")
print(f"Najlepszy model: {best_name.upper()}")
print(f"{'='*60}")
print(f"Test RMSE: {results[best_name]['test_rmse']:.2f}")
print(f"\nNajlepsze parametry:")
for param, value in results[best_name]['best_params'].items():
    print(f"  {param}: {value}")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_name = min(results, key=lambda k: results[k]["test_rmse"])
best_model = results[best_name]["best_model"]

model_filename = f"{user_path}/models/best_regressor_{best_name}_{timestamp}.pkl"
joblib.dump(best_model, model_filename)
print(f"\nNajlepszy model zapisany: {model_filename}")

Classifier - param search

In [ ]:
y = (df["volume_total"] > 10_000).astype(int)
X = df.drop(columns=["volume_total"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y
)

train_class_dist = y_train.value_counts(normalize=True)
test_class_dist = y_test.value_counts(normalize=True)

In [ ]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nRozkład klas (train):")
print(f"  0 (<=10k): {train_class_dist[0]:.1%}")
print(f"  1 (>10k):  {train_class_dist[1]:.1%}")
print(f"\nRozkład klas (test):")
print(f"  0 (<=10k): {test_class_dist[0]:.1%}")
print(f"  1 (>10k):  {test_class_dist[1]:.1%}")

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\npos_weight: {pos_weight:.2f}")

In [ ]:
models_sampling = {
    "xgb_smote_under": (
        [
            ("smote", SMOTE(sampling_strategy=0.3, random_state=2026)),
            ("under", RandomUnderSampler(sampling_strategy=0.7, random_state=2026)),
        ],
        XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=-1,
            random_state=2026,
            scale_pos_weight=pos_weight,
        ),
        {
            "clf__n_estimators": randint(200, 1000),
            "clf__max_depth": randint(3, 10),
            "clf__learning_rate": uniform(0.01, 0.1),
            "clf__subsample": uniform(0.6, 0.4),
            "clf__colsample_bytree": uniform(0.6, 0.4),
            "clf__min_child_weight": randint(1, 10),
            "clf__gamma": uniform(0.0, 2.0),
            "clf__reg_alpha": uniform(0.0, 1.0),
            "clf__reg_lambda": uniform(0.0, 1.0),
        },
    ),
    
    "lgbm_smote_over": (
        [
            ("smote", SMOTE(sampling_strategy=0.3, random_state=2026)),
            ("over", RandomOverSampler(sampling_strategy=0.5, random_state=2026)),
        ],
        LGBMClassifier(
            objective="binary",
            n_jobs=-1,
            random_state=2026,
            is_unbalance=True,
        ),
        {
            "clf__n_estimators": randint(200, 1000),
            "clf__max_depth": randint(-1, 10),
            "clf__learning_rate": uniform(0.01, 0.1),
            "clf__num_leaves": randint(16, 256),
            "clf__subsample": uniform(0.6, 0.4),
            "clf__colsample_bytree": uniform(0.6, 0.4),
            "clf__min_child_samples": randint(10, 60),
            "clf__reg_alpha": uniform(0.0, 1.0),
            "clf__reg_lambda": uniform(0.0, 1.0),
        },
    ),
    
    "cat_smotetomek": (
        [("smotetomek", SMOTETomek(random_state=2026))],
        CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="Recall",
            verbose=0,
            random_seed=2026,
            scale_pos_weight=pos_weight,
        ),
        {
            "clf__depth": randint(4, 10),
            "clf__learning_rate": uniform(0.01, 0.1),
            "clf__n_estimators": randint(200, 1000),
            "clf__l2_leaf_reg": uniform(1.0, 10.0),
        },
    ),
    
    "rf_smoteenn": (
        [("smoteenn", SMOTEENN(random_state=2026))],
        RandomForestClassifier(
            n_jobs=-1,
            random_state=2026,
            class_weight="balanced_subsample",
        ),
        {
            "clf__n_estimators": randint(200, 800),
            "clf__max_depth": randint(3, 20),
            "clf__min_samples_split": randint(2, 20),
            "clf__min_samples_leaf": randint(1, 10),
            "clf__max_features": ["sqrt", "log2", 0.5],
        },
    ),
}

print(f"Zdefiniowano {len(models_sampling)} pipeline'ów klasyfikacji")

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2026)
results_sampling = {}

for name, (samplers, estimator, param_dist) in models_sampling.items():
    print(f"\n{'='*60}\n=== {name.upper()} ===\n{'='*60}")
    
    steps = samplers + [("clf", estimator)]
    pipe = Pipeline(steps=steps)
    
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=30,
        cv=cv,
        scoring="recall",
        n_jobs=-1,
        random_state=2026,
        verbose=1,
    )
    
    search.fit(X_train, y_train)
    
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_pred_proba = best_pipe.predict_proba(X_test)[:, 1]
    
    results_sampling[name] = {
        "search": search,
        "best_pipe": best_pipe,
        "best_params": search.best_params_,
        "cv_recall": search.best_score_,
        "test_recall": recall_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_pred_proba),
    }
    
    print(f"\nBest params: {search.best_params_}")
    print(f"CV Recall: {search.best_score_:.4f}")
    print(f"Test Recall: {results_sampling[name]['test_recall']:.4f}")
    print(f"Test Precision: {results_sampling[name]['test_precision']:.4f}")
    print(f"Test F1: {results_sampling[name]['test_f1']:.4f}")
    print(f"Test AUC: {results_sampling[name]['test_auc']:.4f}")
    
    print(f"\n{classification_report(y_test, y_pred, target_names=['<=10k', '>10k'])}")

In [ ]:
best_name = max(results_sampling, key=lambda k: results_sampling[k]["test_recall"])
best_pipe = results_sampling[best_name]["best_pipe"]

print(f"\n{'='*60}")
print(f"NAJLEPSZY MODEL: {best_name.upper()}")
print(f"{'='*60}")
print(f"Test Recall:    {results_sampling[best_name]['test_recall']:.4f}")
print(f"Test Precision: {results_sampling[best_name]['test_precision']:.4f}")
print(f"Test F1:        {results_sampling[best_name]['test_f1']:.4f}")
print(f"Test AUC:       {results_sampling[best_name]['test_auc']:.4f}")

y_pred_best = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

print(f"\n{'='*60}")
print("Confusion matrix (najlepszy model):")
print(f"{'='*60}")
print(cm)
print(f"\nTrue Negatives:  {cm[0,0]:>6d}")
print(f"False Positives: {cm[0,1]:>6d}")
print(f"False Negatives: {cm[1,0]:>6d}  (Błąd II rodzaju)")
print(f"True Positives:  {cm[1,1]:>6d}")

fnr = cm[1,0] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
print(f"\nFalse Negative Rate (FNR): {fnr:.4f}")

In [ ]:
y_pred_proba_best = best_pipe.predict_proba(X_test)[:, 1]

thresholds = [0.2, 0.3, 0.4, 0.5]

print(f"{'='*80}")
print("Analiza progów decyzyjnych (minimalizacja FN)")
print(f"{'='*80}")
print(f"{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'F1':<10} {'FN':<8} {'FP':<8}")
print(f"{'-'*80}")

for threshold in thresholds:
    y_pred_thresh = (y_pred_proba_best >= threshold).astype(int)
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)
    
    recall_t = recall_score(y_test, y_pred_thresh)
    precision_t = precision_score(y_test, y_pred_thresh)
    f1_t = f1_score(y_test, y_pred_thresh)
    fn_t = cm_thresh[1, 0]
    fp_t = cm_thresh[0, 1]
    
    print(f"{threshold:<12.2f} {recall_t:<10.4f} {precision_t:<12.4f} {f1_t:<10.4f} {fn_t:<8d} {fp_t:<8d}")


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_name = max(results_sampling, key=lambda k: results_sampling[k]["test_recall"])
best_pipe = results_sampling[best_name]["best_pipe"]

pipe_filename = f"{user_path}/models/best_classifier_{best_name}_{timestamp}.pkl"
joblib.dump(best_pipe, pipe_filename)
print(f"\nNajlepszy pipeline zapisany: {pipe_filename}")